# 金科路出发：50分钟“地铁 + 步行”等时圈

这个版本专门用于 **GitHub + Google Colab + Google Sheet**：

- Google Sheet 提供你手工测得的 Apple 地图地铁时间；
- 总时间上限改为 **50分钟**；
- `剩余步行时间 = 50 - Apple地图地铁时间`；
- 只为 `Apple时间 < 50` 的站点请求 ORS 步行等时圈；
- `Apple时间 = 50` 的站点会作为边界站点显示，但不会请求0分钟等时圈；
- ORS缓存与输出保存到 Google Drive，Colab断线后仍可继续。

> 安全提醒：旧 Notebook 中曾直接写入 ORS API key。请在 ORS 后台撤销旧 key，并创建新 key。新 key 只放在 Colab Secrets 中，不要提交到 GitHub。

## 1. 安装依赖

在 Colab 中运行一次即可。安装后若提示重启运行时，再从第2步继续。

In [1]:
%pip install -q "pandas==2.2.2" "requests==2.32.4" geopandas shapely folium tqdm pyogrio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.


## 2. 配置 Google Sheet、Google Drive 与 ORS Secret

### 先在 Colab 中设置 ORS key

1. 打开左侧边栏的 **钥匙图标 / Secrets**；
2. 新建名称：`ORS_API_KEY`；
3. 值填写新的 ORS API key；
4. 打开该 secret 的 Notebook access。

运行本格时，Colab 会要求挂载 Google Drive。缓存和最终结果会保存到：

`MyDrive/Jinke50min/`

In [ ]:
from pathlib import Path
import io
import json
import os
import re
import time

import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import shape
from shapely.ops import unary_union
from tqdm.auto import tqdm
import folium

# -----------------------------
# Google Sheet
# -----------------------------
SHEET_ID = "1zgjzTXIxbgGUOkAIhFW3u7HoV529hx_QZc0advznuC8"
SHEET_GID = "0"
SHEET_CSV_URL = (
    f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/"
    f"export?format=csv&gid={SHEET_GID}"
)

# -----------------------------
# Core settings
# -----------------------------
TOTAL_LIMIT_MIN = 50
ISOCHRONE_INTERVAL_SECONDS = 3.5

# Keep True for the first test.
# After checking the 5 test polygons, change to False.
TEST_MODE = True

TEST_STATIONS = [
    "金科路",
    "人民广场",
    "川沙",
    "徐家汇",
    "野生动物园",
]

# -----------------------------
# Persistent project directory
# -----------------------------
IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Jinke50min")

    try:
        ORS_API_KEY = userdata.get("ORS_API_KEY")
    except Exception:
        ORS_API_KEY = None
else:
    PROJECT_DIR = Path.cwd() / "Jinke50min"
    ORS_API_KEY = os.environ.get("ORS_API_KEY")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = PROJECT_DIR / "ors_cache_50min"
CACHE_DIR.mkdir(exist_ok=True)

COORD_FILE = PROJECT_DIR / "station_coordinates_416.csv"
FAILURE_FILE = PROJECT_DIR / "failed_isochrones_50min.csv"

assert ORS_API_KEY, (
    "没有读取到 ORS_API_KEY。"
    "Colab中请在左侧 Secrets 添加 ORS_API_KEY 并允许 Notebook access。"
)

print("Project directory:", PROJECT_DIR)
print("Google Sheet CSV:", SHEET_CSV_URL)
print("Total limit:", TOTAL_LIMIT_MIN, "minutes")

## 3. 读取 Google Sheet，并计算50分钟内的剩余步行时间

Google Sheet 需要至少包含：

- `station`
- `apple`

本格会自动检查重复站名、空值和无效时间。

In [ ]:
sheet_raw = pd.read_csv(SHEET_CSV_URL)
sheet_raw.columns = [str(col).strip() for col in sheet_raw.columns]

required_columns = {"station", "apple"}
missing_columns = required_columns - set(sheet_raw.columns)

if missing_columns:
    raise ValueError(
        f"Google Sheet缺少字段：{missing_columns}。"
        f"当前字段：{sheet_raw.columns.tolist()}"
    )

df_all = sheet_raw.copy()
df_all["station"] = df_all["station"].astype(str).str.strip()
df_all["apple"] = pd.to_numeric(df_all["apple"], errors="coerce")

invalid_time = df_all[df_all["apple"].isna()]
if not invalid_time.empty:
    raise ValueError(
        "以下站点的apple时间不是数字：\n"
        + invalid_time[["station", "apple"]].to_string(index=False)
    )

duplicates = df_all[df_all["station"].duplicated(keep=False)]
if not duplicates.empty:
    raise ValueError(
        "Google Sheet存在重复站名：\n"
        + duplicates[["station", "apple"]].to_string(index=False)
    )

if (df_all["apple"] < 0).any():
    raise ValueError("apple时间不能小于0。")

# All stations that are reachable by metro in no more than 50 minutes.
reachable_points = df_all[df_all["apple"] <= TOTAL_LIMIT_MIN].copy()

# ORS cannot create a useful 0-second polygon, so only strictly-below-50
# stations enter the polygon request set.
df = df_all[df_all["apple"] < TOTAL_LIMIT_MIN].copy()
df["walk_min"] = TOTAL_LIMIT_MIN - df["apple"]
df["walk_sec"] = (df["walk_min"] * 60).round().astype(int)

boundary_points = df_all[df_all["apple"] == TOTAL_LIMIT_MIN].copy()
outside_points = df_all[df_all["apple"] > TOTAL_LIMIT_MIN].copy()

print(f"Google Sheet total stations: {len(df_all)}")
print(f"Apple time < 50 (request walking polygons): {len(df)}")
print(f"Apple time = 50 (boundary points only): {len(boundary_points)}")
print(f"Apple time > 50 (excluded): {len(outside_points)}")

display(
    df[["ID", "station", "apple", "walk_min", "walk_sec"]]
    .head(15)
)

## 4. 载入416站WGS84坐标并检查匹配

坐标表已嵌入 Notebook，所以从 GitHub 打开到 Colab 后不需要另外上传文件。

同时，本格会把坐标表保存到 Google Drive 项目目录，方便以后单独查看。

In [ ]:
EMBEDDED_COORDINATES_CSV = "station,lon,lat\n金科路,121.597836,31.2064028\n张江高科,121.5833523,31.2039914\n广兰路,121.616873,31.2131129\n龙阳路,121.5534049,31.204691750000002\n唐镇,121.6518673,31.2163796\n世纪公园,121.5466162,31.2114184\n创新中路,121.6698203,31.2155336\n上海科技馆,121.5386617,31.2218914\n华夏东路,121.6767066,31.1988574\n世纪大道,121.52272875,31.230749799999998\n花木路,121.5585663,31.2131883\n芳华路,121.5457808,31.1953299\n芳芯路,121.5550397,31.193162\n迎春路,121.5467582,31.2232213\n川沙,121.6936866,31.1887574\n浦东南路,121.511115,31.2353829\n华夏中路,121.57782125,31.177344599999998\n锦绣路,121.5356397,31.1897121\n北中路,121.5571091,31.1852344\n杨高中路,121.54447195,31.2290013\n陆家嘴,121.49793199999999,31.238544949999998\n凌空路,121.7195131,31.194826\n杨高南路,121.5201071,31.1896768\n民生路,121.54042255,31.237683699999998\n莲溪路,121.5620774,31.1717968\n罗山路,121.5889657,31.155549999999998\n浦东大道,121.5149936,31.2422421\n浦电路,121.5246396,31.222202\n源深体育中心,121.5298954,31.2350364\n商城路,121.512502,31.2325381\n南京东路,121.47968979999999,31.2396888\n高科西路,121.5054186,31.1878023\n御桥,121.5673382,31.161028549999997\n昌邑路,121.53577039999999,31.2460307\n远东大道,121.7509829,31.2014111\n杨树浦路,121.5125161,31.2541829\n蓝村路,121.52354399999999,31.21365595\n小南门,121.4940577,31.2190076\n芳甸路,121.5544198,31.2343994\n人民广场,121.4705235,31.2345013\n云台路,121.4959095,31.1842127\n中科路,121.5975394,31.1810277\n丹阳路,121.5260504,31.2569051\n康桥,121.5630877,31.1365059\n周浦东,121.6027294,31.1123526\n海天三路,121.7926207,31.170532\n上海儿童医学中心,121.5195474,31.2061476\n北洋泾路,121.5477984,31.2411683\n塘桥,121.514395,31.2118426\n大连路,121.50848350000001,31.2600445\n耀华路,121.49025985,31.180391399999998\n南京西路,121.4560232,31.2304667\n源深路,121.5264996,31.2433745\n豫园,121.4835545,31.23069345\n蓝天路,121.57376694999999,31.24308595\n陆家浜路,121.4813149,31.2138337\n学林路,121.6094521,31.185812499999997\n陈春路,121.5538662,31.176986\n周浦,121.563431,31.1167416\n平凉路,121.52253,31.2610517\n临平路,121.4964839,31.2629886\n临沂新村,121.5131584,31.1958017\n南浦大桥,121.4950192,31.2104023\n德平路,121.5600894,31.2476236\n浦东1号2号航站楼,121.8015174,31.15236575\n长清路,121.4828993,31.176278600000003\n大世界,121.4750583,31.22889165\n北蔡,121.5478098,31.182163\n张江路,121.6269441,31.1915332\n秀沿路,121.5942175,31.1403484\n静安寺,121.4421389,31.2249232\n台儿庄路,121.5931398,31.2544453\n天潼路,121.47788495,31.2457666\n马当路,121.4718909,31.2114769\n鹤沙航城,121.6070663,31.0800591\n东明路,121.50702615,31.174807649999998\n江浦公园,121.51921415000001,31.2668975\n繁荣路,121.5661615,31.1056269\n歇浦路,121.547042,31.2526656\n浦三路,121.5350543,31.1531902\n云山路,121.56733905,31.25267375\n海伦路,121.4850594,31.2611414\n西藏南路,121.4851163,31.2036713\n后滩,121.46936475000001,31.175329599999998\n一大会址·黄陂南路,121.46865295,31.22529495\n下南路,121.5362495,31.1814746\n江苏路,121.42662895,31.22149155\n四川北路,121.4794725,31.2537236\n老西门,121.4787259,31.220886\n康新公路,121.6131827,31.1326897\n高青路,121.5113016,31.1614947\n打浦桥,121.46391905,31.20832705\n金桥,121.6072181,31.2631226\n江浦路,121.51434795,31.27652725\n沈梅路,121.5757477,31.0939092\n宝山路,121.4717878,31.2534974\n新闸路,121.4636467,31.2405472\n曲阜路,121.46710505,31.2443051\n金桥路,121.5775913,31.2593229\n鲁班路,121.4705633,31.2010784\n三林东,121.5188767,31.1486404\n龙华中路,121.45219245,31.1861809\n提篮桥,121.5023557,31.2554264\n航头东,121.6132685,31.0572803\n华鹏路,121.5222605,31.1784416\n一大会址·新天地,121.46988139999999,31.218017699999997\n中山公园,121.41130785,31.21999385\n中华艺术宫,121.4893244,31.1872227\n华夏西路,121.5103152,31.1519915\n成山路,121.49239775000001,31.1731258\n黄杨路,121.5862834,31.2359808\n嘉善路,121.456381,31.204453\n抚顺路,121.5113331,31.2860526\n金吉路,121.6247128,31.2666619\n鹤涛路,121.5818149,31.0743335\n迪士尼,121.6637253,31.1434348\n中兴路,121.4643534,31.255078\n汉中路,121.4543465,31.242627\n陕西南路,121.4546929,31.2177863\n上海火车站,121.45219705,31.250383300000003\n博兴路,121.5826511,31.2661406\n大木桥路,121.4592279,31.195805149999998\n东安路,121.4500245,31.19277725\n三林,121.507929,31.1453958\n国际客运中心,121.4939618,31.2521438\n宁国路,121.5283015,31.2706832\n武定路,121.4317639,31.2289\n邮电新村,121.4898084,31.2704149\n淮海中路,121.4598334,31.2218467\n自然博物馆,121.457925,31.2377862\n上南路,121.5019944,31.1511939\n世博大道,121.47859485000001,31.1838692\n杨思,121.4890118,31.1630199\n娄山关路,121.3967585,31.2133704\n云顺路,121.5964372,31.2392917\n常熟路,121.44551115,31.21527385\n昌平路,121.4381473,31.2354606\n下沙,121.5848189,31.0567896\n国权路,121.5062272,31.291247650000003\n世博会博物馆,121.4770974,31.1992369\n肇嘉浜路,121.4458751,31.20095415\n金海路,121.63479715,31.2657431\n中潭路,121.436407,31.2564321\n五莲路,121.5837533,31.274151\n西藏北路,121.4645728,31.265566\n新场,121.6447327,31.0478433\n武宁路,121.4262897,31.23555275\n隆昌路,121.5407047,31.2776265\n上海图书馆,121.4396546,31.2100122\n四平路,121.49706925000001,31.2768594\n东方体育中心,121.47591695,31.155410099999997\n灵岩南路,121.4908095,31.1507738\n浦东足球场,121.6107535,31.2437063\n威宁路,121.3826767,31.2167836\n长寿路,121.43384505,31.242259\n复旦大学,121.4953727,31.2982596\n航头,121.5920502,31.0393345\n交通大学,121.4305565,31.2040947\n隆德路,121.41893015,31.23234265\n鞍山新村,121.5051431,31.275278049999997\n黄兴路,121.5240955,31.2809549\n徐家汇,121.4323653,31.1963162\n顾唐路,121.6522724,31.2682621\n上海体育场,121.43905430000001,31.1861434\n中山北路,121.45447,31.2614891\n巨峰路,121.58468855000001,31.2825586\n虹口足球场,121.47473535,31.27256225\n衡山路,121.4419408,31.2064527\n镇坪路,121.42611735,31.2483655\n东宝兴路,121.475705,31.2618273\n曹杨路,121.41312,31.2411734\n爱国路,121.5482806,31.2819802\n同济大学,121.5020648,31.2844191\n江宁路,121.4398643,31.2461078\n凌兆新村,121.4848877,31.1430265\n龙华,121.4484325,31.1757481\n金粤路,121.6247466,31.2443209\n龙耀路,121.4551734,31.1615085\n北新泾,121.3690974,31.2183955\n上海财经大学,121.4918856,31.3105857\n延安西路,121.4124439,31.21142\n金沙江路,121.40791825,31.2335998\n延吉中路,121.5306115,31.2907317\n上海体育馆,121.432075,31.1841962\n东靖路,121.5846216,31.2928062\n延长路,121.4506814,31.2733596\n曲阳路,121.4857918,31.278536\n野生动物园,121.6948475,31.0524277\n宜山路,121.4227293,31.188519\n民雷路,121.663802,31.2704573\n中宁路,121.4100155,31.2469107\n复兴岛,121.5569642,31.2830231\n虹桥路,121.41683954999999,31.1989589\n芦恒路,121.4938183,31.1202135\n龙漕路,121.4398149,31.1713411\n桂桥路,121.6309638,31.2526202\n岚皋路,121.4171405,31.2580255\n淞虹路,121.3558076,31.2199749\n云锦路,121.4538663,31.1695006\n殷高路,121.4909886,31.3235982\n红宝石路,121.3935935,31.2005785\n长风公园,121.3913599,31.2271864\n黄兴公园,121.5288185,31.2978586\n上海马戏城,121.4463792,31.281219649999997\n五角场,121.5100685,31.2997402\n五洲大道,121.5851041,31.3046922\n申江路,121.6226106,31.2823166\n曹路,121.6784911,31.273409\n桂林路,121.4124743,31.1769519\n上海游泳馆,121.4367543,31.1812817\n枫桥路,121.4057154,31.2443442\n真如,121.4022736,31.25257355\n东陆路,121.5746438,31.2846873\n赤峰路,121.4779161,31.2831792\n宋园路,121.4079018,31.1983438\n浦江镇,121.5018729,31.0986267\n漕宝路,121.42971835,31.1703159\n新村路,121.4182,31.2652861\n惠南,121.7573174,31.056003\n虹桥2号航站楼,121.3216102,31.1961327\n大渡河路,121.3904607,31.23416555\n姚虹路,121.401447,31.1933467\n长江南路,121.48555285,31.3343414\n翔殷路,121.5275779,31.3071133\n汶水路,121.4455881,31.2945322\n江湾体育场,121.5099408,31.3050228\n洲海路,121.585197,31.3143994\n金京路,121.6107908,31.2818724\n漕河泾开发区,121.3930551,31.1724448\n杨高北路,121.598584,31.2822892\n铜川路,121.39252025,31.253037749999997\n大柏树,121.4786881,31.291411\n伊犁路,121.3980259,31.2011073\n桂林公园,121.41431385,31.16937\n江月路,121.5042158,31.0863135\n大华三路,121.4185025,31.2756155\n漕溪路,121.4338149,31.1786074\n吴中路,121.409253,31.1860595\n梅岭北路,121.3921022,31.2457461\n嫩江路,121.5274809,31.316777\n通南路,121.4752177,31.3395488\n上海南站,121.4260241,31.1565603\n彭浦新村,121.4439361,31.3074944\n三门路,121.5038915,31.3152515\n外高桥保税区南,121.5977257,31.3237655\n虹桥火车站,121.3137444,31.1959975\n真光路,121.3797034,31.2550635\n江湾镇,121.4805955,31.3074899\n合川路,121.3801556,31.1684275\n水城路,121.3876028,31.2012461\n联航路,121.5062085,31.0756303\n虹漕路,121.4061275,31.1662502\n上海西站,121.3966434,31.2651065\n惠南东,121.7895151,31.0286244\n行知路,121.4171585,31.2859284\n石龙路,121.4386219,31.1598584\n市光路,121.5276336,31.3247936\n共康路,121.4420271,31.3217011\n锦江乐园,121.4095153,31.1440689\n长江西路,121.4705382,31.3481133\n殷高东路,121.5024309,31.3238015\n七宝,121.3450601,31.1574038\n七莘路,121.3572069,31.1329965\n三林南,121.4820207,31.1280131\n三鲁公路,121.527389,31.056108\n上大路,121.4038689,31.3169815\n上海动物园,121.3634133,31.1920985\n上海国际旅游度假区,121.6759116,31.1585237\n上海大学,121.3839347,31.3223633\n上海松江站,121.2263732,30.9869127\n上海汽车城,121.1762633,31.2873157\n上海赛车场,121.2216614,31.3338783\n东兰路,121.3871482,31.1577076\n东城一路,121.532093,31.03039\n东川路,121.415259,31.0201818\n东方绿舟,121.0148951,31.1005043\n中春路,121.33148209999999,31.1516284\n丰庄,121.3507104,31.2442808\n丰翔路,121.3763776,31.3103095\n临洮路,121.3263327,31.2650033\n临港大道,121.9065641,30.925954\n乐秀路,121.3092454,31.2672762\n九亭,121.3151566,31.139455\n书院,121.846355,30.9615776\n佘山,121.2253618,31.1062374\n元江路,121.427266,31.0647763\n兆丰路,121.150315,31.289048\n光明路,121.117186,31.29621\n共富新村,121.4294324,31.3569381\n刘行,121.3577657,31.3597292\n剑川路,121.4119412,31.0284269\n北桥,121.4054132,31.0470272\n华东理工大学,121.4241171,31.1461157\n华宁路,121.3905802,31.0094092\n南大路,121.3755576,31.3004988\n南翔,121.3185965,31.2988523\n南陈路,121.3936629,31.3235583\n友谊西路,121.4233186,31.3829417\n友谊路,121.4713824,31.4059296\n双柏路,121.4171111,31.0851992\n双江路,121.5351579,31.355695\n古浪路,121.3874843,31.28484495\n向城路,121.5277708,31.2246257\n呼兰路,121.4318178,31.341449150000003\n嘉定北,121.2330115,31.393568\n嘉定新城,121.2498245,31.3320802\n嘉定西,121.223391,31.3790494\n嘉怡路,121.3407328,31.2612945\n嘉松中路,121.21966425,31.1660251\n国家会展中心,121.2909119,31.1932726\n国帆路,121.5087823,31.3416315\n场中路,121.4086121,31.3056307\n基隆路,121.5861815,31.3535319\n外环路,121.3883624,31.1228126\n外高桥保税区北,121.5826967,31.3499474\n大场镇,121.4115867,31.2951955\n奉浦大道,121.4444599,30.9441318\n奉贤新城,121.4915344,30.9163132\n安亭,121.1575136,31.2903851\n定边路,121.3577078,31.2582645\n宝安公路,121.426259,31.3714055\n宝杨路,121.4750613,31.397224\n富锦路,121.4199194,31.3941146\n封浜,121.2925591,31.269456\n康恒路,121.5492931,31.1564411\n康文路,121.4144065,31.3372125\n康桥东,121.611222,31.149121\n张华浜,121.4942142,31.3600102\n徐泾北城,121.2373249,31.1774592\n徐盈路,121.2498298,31.180115\n文井路,121.3761175,31.0055537\n新江湾城,121.5023484,31.3307804\n昌吉东路,121.1958929,31.2955477\n星中路,121.3640688,31.1599995\n春申路,121.381151,31.1001363\n景洪路,121.43810795,31.1127594\n景西路,121.413989,31.101516\n曙建路,121.4130342,31.0926964\n望园路,121.47853315,30.935452599999998\n朱家角,121.0444783,31.1024894\n朱梅路,121.4333425,31.1262981\n李子园,121.3852558,31.2707095\n松江体育中心,121.2262371,31.0179937\n松江大学城,121.2281724,31.0561736\n松江新城,121.2262586,31.0322111\n桃浦新村,121.3450707,31.2834828\n武威东路,121.387403,31.2771479\n武威路,121.3600778,31.2783683\n殷高西路,121.4804318,31.3219502\n水产路,121.4836827,31.3832818\n永德路,121.4385487,31.041227\n汇臻路,121.524558,31.025245\n汇金路,121.1474612,31.1631754\n江川路,121.4187782,31.0074377\n江杨北路,121.4352116,31.4096652\n沈杜公路,121.5079105,31.0634917\n泗泾,121.2558797,31.1204097\n洞泾,121.2260333,31.0865443\n浦航路,121.530591,31.040993\n淀山湖大道,121.0774053,31.1361923\n淞发路,121.4959577,31.3471701\n淞滨路,121.4882612,31.3730074\n港城路,121.5707186,31.3552417\n滴水湖,121.9257641,30.9093062\n漕盈路,121.0919563,31.162321\n潘广路,121.3513174,31.3660078\n爱辉路,121.4446366,31.34601\n环城东路,121.4588819,30.9330926\n白银路,121.240882,31.3472801\n真北路,121.3775593,31.2340365\n真新新村,121.3674467,31.2572465\n祁华路,121.3688336,31.3241778\n祁安路,121.3802641,31.2934242\n祁连山南路,121.3626932,31.2394924\n祁连山路,121.3722123,31.273409\n紫竹高新区,121.4462399,31.025917\n紫藤路,121.3609817,31.1717823\n罗南新村,121.3527972,31.3905792\n罗秀路,121.4311874,31.1337004\n美兰湖,121.3452915,31.4036756\n航中路,121.3491547,31.1671579\n航津路,121.5897812,31.3375153\n花桥,121.104407,31.29879\n莘庄,121.38074585000001,31.11308195\n莲花路,121.3982663,31.1328603\n萧塘,121.4374021,30.9678558\n虹桥1号航站楼,121.3427146,31.1932684\n虹梅南路,121.4283228,31.105705\n虹梅路,121.3926923,31.1622429\n虹莘路,121.3758226,31.1395765\n蟠祥路·国家会计学院,121.2817657,31.1834815\n蟠龙路,121.2742138,31.1884239\n西岑,120.9598944,31.0726117\n西渡,121.42764015,30.9913353\n赵巷,121.1877018,31.1630876\n通河新村,121.4369064,31.3329517\n醉白池,121.2244749,31.0028512\n金平路,121.4054914,31.0132016\n金沙江西路,121.3306649,31.2430562\n金海湖,121.4877261,30.931089800000002\n金运路,121.3148634,31.2427956\n铁力路,121.4567134,31.4099505\n银都路,121.3856299,31.0912508\n锦秋路,121.3773156,31.322188\n闵瑞路,121.530323,31.047956\n闵行开发区,121.3650334,31.0025914\n陈翔公路,121.3023394,31.3083445\n青浦新城,121.1210999,31.1607843\n顾戴路,121.3874729,31.1432364\n顾村公园,121.3691844,31.34570335\n颛桥,121.397231,31.068916\n马陆,121.2723972,31.3215952\n高桥,121.5587311,31.3547358\n高桥西,121.5450534,31.3536141\n龙柏新村,121.3659379,31.1792624\n龙溪路,121.3757005,31.1963362\n"

# Save a transparent standalone copy to Drive.
if not COORD_FILE.exists():
    COORD_FILE.write_text(
        EMBEDDED_COORDINATES_CSV,
        encoding="utf-8-sig",
    )

coordinates = pd.read_csv(
    io.StringIO(EMBEDDED_COORDINATES_CSV)
)
coordinates["station"] = (
    coordinates["station"]
    .astype(str)
    .str.strip()
)
coordinates["lon"] = pd.to_numeric(
    coordinates["lon"],
    errors="coerce",
)
coordinates["lat"] = pd.to_numeric(
    coordinates["lat"],
    errors="coerce",
)

work_all = df_all.merge(
    coordinates,
    on="station",
    how="left",
    validate="one_to_one",
)

missing_coordinates = work_all[
    work_all[["lon", "lat"]].isna().any(axis=1)
]

if not missing_coordinates.empty:
    raise ValueError(
        "以下Google Sheet站点没有坐标：\n"
        + missing_coordinates[["station"]].to_string(index=False)
    )

work = df.merge(
    coordinates,
    on="station",
    how="left",
    validate="one_to_one",
)

reachable_with_coords = reachable_points.merge(
    coordinates,
    on="station",
    how="left",
    validate="one_to_one",
)

print(f"Coordinates loaded: {len(coordinates)}")
print(f"Processable stations with coordinates: {len(work)}")
display(work[["station", "apple", "walk_min", "lon", "lat"]].head(15))

## 5. 检查坐标地图

所有点应落在上海、花桥及邻近区域。红色大点是金科路；蓝色点是Apple时间不超过50分钟的站点。

In [ ]:
check_map = folium.Map(
    location=[31.23, 121.47],
    zoom_start=10,
    tiles="OpenStreetMap",
)

for _, row in reachable_with_coords.iterrows():
    is_origin = row["station"] == "金科路"

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7 if is_origin else 3,
        tooltip=(
            f'{row["station"]}<br>'
            f'Apple地铁：{row["apple"]:.0f}分钟'
        ),
        color="#b2182b" if is_origin else "#2166ac",
        fill=True,
        fill_color="#ef8a62" if is_origin else "#67a9cf",
        fill_opacity=0.9,
    ).add_to(check_map)

check_map

## 6. 请求 ORS 步行等时圈

首次运行保持：

```python
TEST_MODE = True
```

只测试5个站。地图检查通过后，把第2步中的 `TEST_MODE` 改成 `False`，重新运行第2、3、4、6步，然后继续第7和第8步。

ORS Standard计划每分钟20次请求，本格用3.5秒间隔。按当前Google Sheet，50分钟方案会处理约175个站，完整首次运行大约需要10—15分钟。

In [ ]:
ISOCHRONE_URL = (
    "https://api.openrouteservice.org/"
    "v2/isochrones/foot-walking"
)


def safe_filename(text: str) -> str:
    return re.sub(
        r'[^0-9A-Za-z\u4e00-\u9fff._-]+',
        "_",
        str(text),
    )


def cache_path_for(station: str, walk_sec: int) -> Path:
    return (
        CACHE_DIR
        / f"{safe_filename(station)}_{int(walk_sec)}s.json"
    )


def retry_wait_seconds(response, attempt: int) -> float:
    retry_after = response.headers.get("Retry-After")

    if retry_after:
        try:
            return max(float(retry_after), 10.0)
        except ValueError:
            pass

    return min(30 * (attempt + 1), 300)


def request_isochrone(
    station: str,
    lon: float,
    lat: float,
    walk_sec: int,
    apple_min: float,
    session: requests.Session,
    max_retries: int = 8,
) -> dict:
    headers = {
        "Authorization": ORS_API_KEY,
        "Content-Type": "application/json; charset=utf-8",
        "Accept": "application/geo+json, application/json",
    }

    payload = {
        "locations": [[float(lon), float(lat)]],
        "range": [int(walk_sec)],
        "range_type": "time",
        "attributes": ["area", "reachfactor"],
        "smoothing": 20,
    }

    for attempt in range(max_retries):
        response = session.post(
            ISOCHRONE_URL,
            headers=headers,
            json=payload,
            timeout=180,
        )

        if response.status_code == 200:
            result = response.json()

            if not result.get("features"):
                raise RuntimeError(
                    f"{station}: ORS returned no polygon features"
                )

            # Add origin metadata to each feature for later auditing.
            for feature in result["features"]:
                properties = feature.setdefault("properties", {})
                properties.update(
                    {
                        "station": station,
                        "apple_min": float(apple_min),
                        "walk_sec": int(walk_sec),
                        "total_limit_min": TOTAL_LIMIT_MIN,
                    }
                )

            return result

        if response.status_code == 429:
            wait_seconds = retry_wait_seconds(
                response,
                attempt,
            )
            print(
                f"\n{station}: HTTP 429. "
                f"Waiting {wait_seconds:.0f} seconds..."
            )
            time.sleep(wait_seconds)
            continue

        if 500 <= response.status_code < 600:
            wait_seconds = min(
                20 * (attempt + 1),
                180,
            )
            print(
                f"\n{station}: ORS server error "
                f"{response.status_code}. "
                f"Waiting {wait_seconds} seconds..."
            )
            time.sleep(wait_seconds)
            continue

        raise RuntimeError(
            f"{station}: HTTP {response.status_code} — "
            f"{response.text[:500]}"
        )

    raise RuntimeError(
        f"{station}: failed after {max_retries} retries"
    )


processable = work.copy()

if TEST_MODE:
    run_df = processable[
        processable["station"].isin(TEST_STATIONS)
    ].copy()

    missing_tests = (
        set(TEST_STATIONS)
        - set(run_df["station"])
    )

    if missing_tests:
        raise ValueError(
            f"Test stations unavailable: {sorted(missing_tests)}"
        )
else:
    run_df = processable.copy()

print(
    run_df[
        [
            "station",
            "apple",
            "walk_min",
            "lon",
            "lat",
        ]
    ]
)

print(f"All processable stations: {len(processable)}")
print(f"Selected for this run: {len(run_df)}")
print(f"TEST_MODE: {TEST_MODE}")

session = requests.Session()
failed = []
last_request_started = None

for _, row in tqdm(
    run_df.iterrows(),
    total=len(run_df),
    desc="Walking isochrones",
):
    station = str(row["station"])
    walk_seconds = int(row["walk_sec"])
    cache_file = cache_path_for(
        station,
        walk_seconds,
    )

    if cache_file.exists():
        continue

    if last_request_started is not None:
        elapsed = (
            time.monotonic()
            - last_request_started
        )
        remaining_wait = (
            ISOCHRONE_INTERVAL_SECONDS
            - elapsed
        )

        if remaining_wait > 0:
            time.sleep(remaining_wait)

    try:
        last_request_started = time.monotonic()

        result = request_isochrone(
            station=station,
            lon=row["lon"],
            lat=row["lat"],
            walk_sec=walk_seconds,
            apple_min=row["apple"],
            session=session,
        )

        cache_file.write_text(
            json.dumps(
                result,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

    except Exception as exc:
        error_text = str(exc)

        print(
            f"\n{station} failed: "
            f"{error_text}"
        )

        failed.append(
            {
                "station": station,
                "lon": row["lon"],
                "lat": row["lat"],
                "apple": row["apple"],
                "walk_sec": walk_seconds,
                "walk_min": walk_seconds / 60,
                "error": error_text,
            }
        )

if failed:
    pd.DataFrame(failed).to_csv(
        FAILURE_FILE,
        index=False,
        encoding="utf-8-sig",
    )
else:
    if FAILURE_FILE.exists():
        FAILURE_FILE.unlink()

expected_cache_files = {
    cache_path_for(
        row["station"],
        int(row["walk_sec"]),
    ).name
    for _, row in processable.iterrows()
}

actual_expected_cache_files = {
    path.name
    for path in CACHE_DIR.glob("*.json")
    if path.name in expected_cache_files
}

print()
print("Finished this run.")
print(
    "Expected full-run cache files:",
    len(expected_cache_files),
)
print(
    "Available current 50-minute cache files:",
    len(actual_expected_cache_files),
)
print("Failures during this run:", len(failed))

if failed:
    display(pd.DataFrame(failed))

## 7. 合并当前有效的50分钟等时圈

- 测试模式下，只会合并已经生成的5个测试站；
- 全量模式完成后，会合并全部约175个站；
- 只读取当前50分钟方案预期的缓存文件，不会误用旧60分钟缓存。

In [ ]:
records = []
used_cache_files = []

for _, row in processable.iterrows():
    cache_file = cache_path_for(
        row["station"],
        int(row["walk_sec"]),
    )

    if not cache_file.exists():
        continue

    data = json.loads(
        cache_file.read_text(
            encoding="utf-8"
        )
    )

    for feature in data.get("features", []):
        props = (
            feature.get("properties", {})
            .copy()
        )
        props["cache_file"] = cache_file.name
        props["geometry"] = shape(
            feature["geometry"]
        )
        records.append(props)

    used_cache_files.append(cache_file.name)

if not records:
    raise RuntimeError(
        "没有找到当前50分钟方案的等时圈缓存。"
        "请先运行第6步。"
    )

iso_gdf = gpd.GeoDataFrame(
    records,
    geometry="geometry",
    crs="EPSG:4326",
)

individual_file = (
    PROJECT_DIR
    / "individual_station_isochrones_50min.geojson"
)

iso_gdf.to_file(
    individual_file,
    driver="GeoJSON",
)

try:
    union_geom = iso_gdf.geometry.union_all()
except AttributeError:
    union_geom = unary_union(
        iso_gdf.geometry.tolist()
    )

union_gdf = gpd.GeoDataFrame(
    [
        {
            "origin": "金科路",
            "total_limit_min": TOTAL_LIMIT_MIN,
            "station_polygon_count": len(iso_gdf),
            "geometry": union_geom,
        }
    ],
    crs="EPSG:4326",
)

union_file = (
    PROJECT_DIR
    / "jinke_50min_metro_walk_union.geojson"
)

union_gdf.to_file(
    union_file,
    driver="GeoJSON",
)

missing_cache_count = (
    len(processable)
    - len(set(used_cache_files))
)

print("Polygons included:", len(iso_gdf))
print("Missing full-run caches:", missing_cache_count)
print("Individual output:", individual_file)
print("Union output:", union_file)

## 8. 生成交互式50分钟地图

地图只显示 Apple 时间不超过50分钟的地铁站。  
Apple时间等于50分钟的站点显示为边界点，但不会产生步行多边形。

In [ ]:
m = folium.Map(
    location=[31.23, 121.47],
    zoom_start=10,
    tiles="OpenStreetMap",
)

folium.GeoJson(
    union_gdf.to_json(),
    name="50分钟地铁+步行合并范围",
    style_function=lambda _: {
        "color": "#d7301f",
        "weight": 2,
        "fillColor": "#fc8d59",
        "fillOpacity": 0.28,
    },
).add_to(m)

for _, row in reachable_with_coords.iterrows():
    is_origin = row["station"] == "金科路"
    remaining_walk = max(
        0,
        TOTAL_LIMIT_MIN - row["apple"],
    )

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7 if is_origin else 3,
        tooltip=(
            f'{row["station"]}<br>'
            f'Apple地铁：{row["apple"]:.0f}分钟<br>'
            f'剩余步行：{remaining_walk:.0f}分钟'
        ),
        color=(
            "#b8860b"
            if is_origin
            else "#b2182b"
        ),
        fill=True,
        fill_color=(
            "#ffd700"
            if is_origin
            else "#ef8a62"
        ),
        fill_opacity=0.95,
    ).add_to(m)

folium.LayerControl().add_to(m)

html_file = (
    PROJECT_DIR
    / "jinke_50min_metro_walk_map.html"
)

m.save(html_file)

print("HTML output:", html_file)
m

## 9. 输出位置

结果保存在 Google Drive：

`MyDrive/Jinke50min/`

主要文件：

- `ors_cache_50min/`：每个站的ORS缓存；
- `station_coordinates_416.csv`：416站坐标；
- `individual_station_isochrones_50min.geojson`：单站等时圈；
- `jinke_50min_metro_walk_union.geojson`：最终合并范围；
- `jinke_50min_metro_walk_map.html`：交互式地图；
- `failed_isochrones_50min.csv`：失败站点，仅失败时存在。

重新运行时会复用Google Drive中的缓存。